# True Virtual Base Station RINEX Generator

This notebook generates a **coordinate-shifted virtual base station** RINEX observation file from a real base `.obs` + `.nav` pair.

Unlike a header-only edit, this applies a **true geometric correction** to each observation so downstream RTK/PPK software treats the base as physically located at the new coordinates.

For each satellite and epoch, we compute the single-difference geometric correction:

\[
\Deltaho = \|\mathbf{s} - \mathbf{x}_{new}\| - \|\mathbf{s} - \mathbf{x}_{real}\|
\]

Then apply:

- pseudorange: `P_corrected = P_real + Δρ`
- carrier phase (cycles): `L_corrected = L_real + Δρ / λ`

In [ ]:
!pip install georinex numpy xarray

import os
from pathlib import Path
import numpy as np
import xarray as xr
import georinex as gr

## Parameters

Set input paths and coordinates here. This cell is tagged for parameterization tools (e.g., papermill).

Use either LLH (`*_llh`) **or** XYZ (`*_xyz`) for each position. If both are provided, LLH is used.

In [ ]:
# Input RINEX files
obs_path = 'real_base.obs'
nav_path = 'real_base.nav'

# Real base coordinates (choose one representation)
real_llh = None                      # Example: (62.0123, -6.7890, 45.2) [lat deg, lon deg, h m]
real_xyz = (0.0, 0.0, 0.0)           # Example: (X, Y, Z) ECEF meters

# New virtual base coordinates (choose one representation)
new_llh = None                       # Example: (62.0200, -6.8000, 48.0)
new_xyz = (10.0, 0.0, 0.0)           # Example: (X, Y, Z) ECEF meters

# Output file
out_path = 'virtual_base.obs'

## WGS84 helper: LLH to ECEF XYZ

In [ ]:
WGS84_A = 6378137.0
WGS84_F = 1 / 298.257223563
WGS84_E2 = WGS84_F * (2 - WGS84_F)
C = 299792458.0  # m/s

def llh2xyz(lat_deg, lon_deg, h_m):
    lat = np.radians(lat_deg)
    lon = np.radians(lon_deg)
    N = WGS84_A / np.sqrt(1 - WGS84_E2 * np.sin(lat) ** 2)
    x = (N + h_m) * np.cos(lat) * np.cos(lon)
    y = (N + h_m) * np.cos(lat) * np.sin(lon)
    z = (N * (1 - WGS84_E2) + h_m) * np.sin(lat)
    return np.array([x, y, z], dtype=float)

def resolve_xyz(llh, xyz, name):
    if llh is not None:
        if len(llh) != 3:
            raise ValueError(f'{name}_llh must have 3 elements (lat, lon, h).')
        return llh2xyz(*llh)
    if xyz is None or len(xyz) != 3:
        raise ValueError(f'Provide either {name}_llh or {name}_xyz with 3 values.')
    return np.array(xyz, dtype=float)

## Satellite position propagation (GPS broadcast ephemeris)

The function below implements ICD-GPS-200-style broadcast ephemeris propagation for GPS satellites (`Gxx`).

> Limitation: this notebook version is GPS-only for orbit propagation. GLONASS and Galileo require different ephemeris/orbit models.

In [ ]:
def _as_float(eph, key):
    v = eph[key].values
    if np.size(v) == 0:
        raise KeyError(key)
    return float(np.asarray(v).squeeze())

def sat_positions_at(nav_ds, sv, times):
    """Compute ECEF satellite positions for a GPS SV at given times."""
    GM = 3.986005e14
    OMEGA_E_DOT = 7.2921151467e-5

    nav_sv = nav_ds.sel(sv=sv)
    toc_times = nav_sv.time.values.astype('datetime64[s]')
    positions = np.full((len(times), 3), np.nan, dtype=float)

    for i, t in enumerate(times):
        t = np.datetime64(t, 's')
        idxs = np.where(toc_times <= t)[0]
        idx = int(idxs[-1]) if len(idxs) else 0
        eph = nav_sv.isel(time=idx)

        try:
            sqrtA = _as_float(eph, 'sqrtA')
            e = _as_float(eph, 'Eccentricity')
            i0 = _as_float(eph, 'Io')
            omega0 = _as_float(eph, 'Omega0')
            omega = _as_float(eph, 'omega')
            M0 = _as_float(eph, 'M0')
            deltaN = _as_float(eph, 'DeltaN')
            idot = _as_float(eph, 'IDOT')
            omegaDot = _as_float(eph, 'OmegaDot')
            cuc = _as_float(eph, 'Cuc')
            cus = _as_float(eph, 'Cus')
            crc = _as_float(eph, 'Crc')
            crs = _as_float(eph, 'Crs')
            cic = _as_float(eph, 'Cic')
            cis = _as_float(eph, 'Cis')
            toe = _as_float(eph, 'Toe')
        except KeyError as ex:
            raise RuntimeError(f'Missing ephemeris field {ex} for {sv}') from ex

        A = sqrtA ** 2
        n0 = np.sqrt(GM / A ** 3)
        n = n0 + deltaN
        tk = float((t - toc_times[idx]) / np.timedelta64(1, 's'))

        Mk = M0 + n * tk
        Ek = Mk
        for _ in range(12):
            Ek = Mk + e * np.sin(Ek)

        vk = np.arctan2(np.sqrt(1 - e * e) * np.sin(Ek), np.cos(Ek) - e)
        phik = vk + omega

        duk = cus * np.sin(2 * phik) + cuc * np.cos(2 * phik)
        drk = crs * np.sin(2 * phik) + crc * np.cos(2 * phik)
        dik = cis * np.sin(2 * phik) + cic * np.cos(2 * phik)

        uk = phik + duk
        rk = A * (1 - e * np.cos(Ek)) + drk
        ik = i0 + idot * tk + dik

        xk_p = rk * np.cos(uk)
        yk_p = rk * np.sin(uk)

        omegak = omega0 + (omegaDot - OMEGA_E_DOT) * tk - OMEGA_E_DOT * toe

        xk = xk_p * np.cos(omegak) - yk_p * np.cos(ik) * np.sin(omegak)
        yk = xk_p * np.sin(omegak) + yk_p * np.cos(ik) * np.cos(omegak)
        zk = yk_p * np.sin(ik)

        positions[i, :] = (xk, yk, zk)

    return positions

## Geometric range and correction model

In [ ]:
def geometric_range(sat_xyz, station_xyz):
    return np.linalg.norm(sat_xyz - station_xyz[None, :], axis=1)

def build_range_corrections(obs_ds, nav_ds, real_xyz, new_xyz):
    """Return dict: sv -> delta_rho(time) in meters."""
    times = obs_ds.time.values
    gps_svs = [sv for sv in obs_ds.sv.values if str(sv).startswith('G')]
    nav_svs = set(str(sv) for sv in nav_ds.sv.values)

    corrections = {}
    for sv in gps_svs:
        if str(sv) not in nav_svs:
            continue
        try:
            sat_xyz = sat_positions_at(nav_ds, sv, times)
        except Exception as ex:
            print(f'Skipping {sv}: {ex}')
            continue

        valid = np.all(np.isfinite(sat_xyz), axis=1)
        if not np.any(valid):
            continue

        drho = np.full(len(times), np.nan, dtype=float)
        rho_real = geometric_range(sat_xyz[valid], real_xyz)
        rho_new = geometric_range(sat_xyz[valid], new_xyz)
        drho[valid] = rho_new - rho_real
        corrections[str(sv)] = xr.DataArray(drho, dims=('time',), coords={'time': obs_ds.time})

    return corrections

## Load observation and navigation data

In [ ]:
def load_data(obs_path, nav_path):
    obs_ds = gr.load(obs_path)
    nav_ds = gr.load(nav_path)

    print('Loaded observation dataset:')
    print(f'  Epochs: {obs_ds.sizes.get("time", 0)}')
    print(f'  SVs:    {obs_ds.sizes.get("sv", 0)}')
    if obs_ds.sizes.get('time', 0) > 0:
        print(f'  Time range: {obs_ds.time.values[0]} -> {obs_ds.time.values[-1]}')

    print('Loaded navigation dataset:')
    print(f'  Epochs: {nav_ds.sizes.get("time", 0)}')
    print(f'  SVs:    {nav_ds.sizes.get("sv", 0)}')

    return obs_ds, nav_ds

## Apply geometric corrections to observables

- Pseudorange variables: names starting with `C` or `P` (meters) → add `Δρ`
- Carrier phase variables: names starting with `L` (cycles) → add `Δρ / λ`

In [ ]:
FREQ_BY_BAND_HZ = {
    '1': 1575.42e6,  # L1
    '2': 1227.60e6,  # L2
    '5': 1176.45e6,  # L5
}

def wavelength_for_obs_var(var_name):
    # var examples: L1C, L2W, L5Q
    if len(var_name) > 1 and var_name[1].isdigit():
        band = var_name[1]
        if band in FREQ_BY_BAND_HZ:
            return C / FREQ_BY_BAND_HZ[band]
    # fallback to L1 wavelength
    return C / FREQ_BY_BAND_HZ['1']

def apply_corrections(obs_ds, corrections):
    out = obs_ds.copy(deep=True)
    data_vars = list(out.data_vars)

    corrected_sv_count = 0
    for sv in out.sv.values:
        sv_key = str(sv)
        if sv_key not in corrections:
            continue

        drho = corrections[sv_key]
        corrected_sv_count += 1

        for var in data_vars:
            arr = out[var]
            if 'sv' not in arr.dims or 'time' not in arr.dims:
                continue

            if var.startswith('C') or var.startswith('P'):
                out[var].loc[dict(sv=sv)] = arr.sel(sv=sv) + drho
            elif var.startswith('L'):
                wavelength = wavelength_for_obs_var(var)
                out[var].loc[dict(sv=sv)] = arr.sel(sv=sv) + (drho / wavelength)

    return out, corrected_sv_count

## Write corrected observation file

`georinex` does not currently provide a native RINEX observation writer, so this notebook includes a simplified RINEX-3-style writer:

1. Copy header from source `.obs`
2. Replace `APPROX POSITION XYZ`
3. Write corrected epoch/SV observation values

In [ ]:
def _format_approx_position_line(new_xyz):
    return f"{new_xyz[0]:14.4f}{new_xyz[1]:14.4f}{new_xyz[2]:14.4f}{'':18s}APPROX POSITION XYZ\n"

def write_rinex_obs(obs_ds, template_obs_path, out_path, new_xyz):
    template_obs_path = Path(template_obs_path)
    out_path = Path(out_path)

    with template_obs_path.open('r', encoding='utf-8', errors='ignore') as f:
        header = []
        for line in f:
            header.append(line)
            if 'END OF HEADER' in line:
                break

    new_header = []
    replaced = False
    for line in header:
        if 'APPROX POSITION XYZ' in line:
            new_header.append(_format_approx_position_line(new_xyz))
            replaced = True
        else:
            new_header.append(line)

    if not replaced:
        insert_idx = next((i for i, ln in enumerate(new_header) if 'END OF HEADER' in ln), len(new_header))
        new_header.insert(insert_idx, _format_approx_position_line(new_xyz))

    data_vars = [v for v in obs_ds.data_vars if {'time', 'sv'}.issubset(set(obs_ds[v].dims))]
    times = obs_ds.time.values
    svs = [str(s) for s in obs_ds.sv.values]

    with out_path.open('w', encoding='utf-8') as out:
        out.writelines(new_header)

        for t in times:
            valid_svs = []
            for sv in svs:
                has_valid = False
                for var in data_vars:
                    v = obs_ds[var].sel(time=t, sv=sv).values
                    if np.isfinite(v):
                        has_valid = True
                        break
                if has_valid:
                    valid_svs.append(sv)

            ts = np.datetime64(t, 'us')
            ts_s = str(ts)
            date_part, time_part = ts_s.split('T')
            y, m, d = map(int, date_part.split('-'))
            hh, mm, ss_frac = time_part.split(':')
            hh = int(hh)
            mm = int(mm)
            sec = float(ss_frac)

            out.write(f"> {y:4d} {m:2d} {d:2d} {hh:2d} {mm:2d} {sec:10.7f}  0{len(valid_svs):3d}\n")

            for sv in valid_svs:
                vals = []
                for var in data_vars:
                    v = obs_ds[var].sel(time=t, sv=sv).values
                    if np.isfinite(v):
                        vals.append(f"{float(v):14.3f}  ")
                    else:
                        vals.append(' ' * 16)
                out.write(f"{sv}{''.join(vals)}\n")

## Run pipeline end-to-end

In [ ]:
real_xyz_vec = resolve_xyz(real_llh, real_xyz, 'real')
new_xyz_vec = resolve_xyz(new_llh, new_xyz, 'new')

shift = new_xyz_vec - real_xyz_vec
print(f'Real base XYZ:    {real_xyz_vec}')
print(f'Virtual base XYZ: {new_xyz_vec}')
print(f'Shift vector (m): {shift}')
print(f'Shift magnitude:  {np.linalg.norm(shift):.3f} m')

if not Path(obs_path).exists() or not Path(nav_path).exists():
    print('\nInput files not found. Update obs_path/nav_path in the parameters cell, then run this cell again.')
else:
    obs_ds, nav_ds = load_data(obs_path, nav_path)

    corrections = build_range_corrections(obs_ds, nav_ds, real_xyz_vec, new_xyz_vec)
    print(f'Computed corrections for {len(corrections)} GPS satellites.')

    obs_corr, corrected_sv_count = apply_corrections(obs_ds, corrections)
    print(f'Applied corrections to {corrected_sv_count} satellites across available observables.')

    write_rinex_obs(obs_corr, obs_path, out_path, new_xyz_vec)
    print(f'Wrote corrected virtual-base observation file: {out_path}')

## Caveats and limitations

- **GPS-only broadcast propagation** in this version (`G*` satellites). To extend to GLONASS/Galileo, add constellation-specific ephemeris propagation.
- **Simplified writer**: `georinex` has no native RINEX OBS writer; this notebook writes a simplified RINEX-3-style body (no LLI/signal-strength flags). For production-grade I/O, prefer RTKLIB workflows (e.g., `convbin`) or a dedicated GNSS library/writer.
- **Pseudorange/phase only** are corrected. Doppler (`D*`) requires separate range-rate correction.
- **No tropo/iono path-delta modeling**. For small coordinate shifts (meters to a few km), this is usually acceptable; larger shifts should include atmospheric delta models.